In [1]:
# TITULO: Entrenamiento Comparativo - Detector de Placas
import os
from ultralytics import YOLO
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt

# Rutas de los datasets generados
YAML_CLEAN = '../../datasets/02_placas/data.yaml'
YAML_BASELINE = '../../datasets/02_placas_baseline/data.yaml'

# Ruta de salida de modelos
MODELS_DIR = '../../models/02_placas'

# Fecha para versionado
DATE_STR = datetime.now().strftime('%Y%m%d')

print("Configuracion lista.")

Configuracion lista.


In [2]:
# Función para obtener el optimizador real cuando se usa 'auto'
def obtener_optimizador_real(modelo):
    """
    Recupera el nombre real del optimizador cuando la configuración es 'auto'.
    """
    try:
        # 1. Si el entrenamiento acaba de terminar y el objeto sigue en memoria RAM
        if hasattr(modelo, 'trainer') and modelo.trainer and hasattr(modelo.trainer, 'optimizer'):
            # El optimizador es un objeto (ej. <torch.optim.sgd.SGD object at 0x...>)
            # Obtenemos su nombre de clase real
            opt_obj = modelo.trainer.optimizer
            nombre_real = type(opt_obj).__name__
            
            # También podemos sacar el Learning Rate real final
            lr_final = opt_obj.param_groups[0]['lr']
            
            print(f"Decisión de 'Auto':")
            print(f"   • Optimizador:   {nombre_real}") # Dirá 'SGD' o 'AdamW'
            print(f"   • Learning Rate: {lr_final:.6f}")
            return

        # 2. Si el modelo fue cargado desde disco (.pt) y no hay trainer en memoria
        # Buscamos en los metadatos internos del archivo
        if hasattr(modelo, 'ckpt') and modelo.ckpt:
            train_args = modelo.ckpt.get('train_args', {})
            # A veces aquí también dice 'auto', en cuyo caso la única verdad está en los logs de texto
            print(f"Configuración guardada: {train_args.get('optimizer', 'Desconocido')}")
            print("Si aquí dice 'auto', por favor revisa el archivo '/runs/.../train/main.log'")

    except Exception as e:
        print(f"No se pudo recuperar automáticamente: {e}")

In [ ]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v8n_baseline_tl = f"01_v8n_baseline_tl"

print(f"Iniciando entrenamiento: {run_name_v8n_baseline_tl}")

# Cargar modelo Nano pre-entrenado
model_v8n_baseline_tl = YOLO('yolov8n.pt')

results_v8n_baseline_tl = model_v8n_baseline_tl.train(
    data=YAML_BASELINE,
    project=MODELS_DIR,
    name=run_name_v8n_baseline_tl,

    epochs=100,            # Ajustable
    patience=20,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=640,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=True,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 01_v8n_baseline_tl
New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas_baseline/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.

In [ ]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v8n_baseline_tl.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v8n_baseline_tl)}")
print(f"Learning Rate inicial: {model_v8n_baseline_tl.trainer.args.lr0}")

Batch Size: 49
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000040
Optimizer: None
Learning Rate inicial: 0.01


In [ ]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v8n_baseline_tl_300 = f"01_v8n_baseline_tl_300"

print(f"Iniciando entrenamiento: {run_name_v8n_baseline_tl_300}")

# Cargar modelo Nano pre-entrenado
model_v8n_baseline_tl_300 = YOLO('yolov8n.pt')

results_v8n_baseline_tl_300 = model_v8n_baseline_tl_300.train(
    data=YAML_BASELINE,
    project=MODELS_DIR,
    name=run_name_v8n_baseline_tl_300,

    epochs=300,            # Ajustable
    patience=50,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=640,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=True,
    optimizer='auto',
    verbose=True
)

In [ ]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v8n_baseline_tl_300.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v8n_baseline_tl_300)}")
print(f"Learning Rate inicial: {model_v8n_baseline_tl_300.trainer.args.lr0}")

In [3]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v8n_baseline_no_tl = f"02_v8n_baseline_no_tl"

print(f"Iniciando entrenamiento: {run_name_v8n_baseline_no_tl}")

# Cargar modelo Nano pre-entrenado
model_v8n_baseline_no_tl = YOLO('yolov8n.yaml')

results_v8n_baseline_no_tl = model_v8n_baseline_no_tl.train(
    data=YAML_BASELINE,
    project=MODELS_DIR,
    name=run_name_v8n_baseline_no_tl,

    epochs=300,            # Ajustable
    patience=20,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=640,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=False,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 02_v8n_baseline_no_tl
New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas_baseline/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, moment

In [4]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v8n_baseline_no_tl.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v8n_baseline_no_tl)}")
print(f"Learning Rate inicial: {model_v8n_baseline_no_tl.trainer.args.lr0}")

Batch Size: 49
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000548
Optimizer: None
Learning Rate inicial: 0.01


In [3]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v8n_resplit_tl = f"03_v8n_resplit_tl"

print(f"Iniciando entrenamiento: {run_name_v8n_resplit_tl}")

model_v8n_resplit_tl = YOLO('yolov8n.pt')

results_v8n_resplit_tl = model_v8n_resplit_tl.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_v8n_resplit_tl,

    epochs=100,            # Ajustable
    patience=20,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=640,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=True,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 03_v8n_resplit_tl
New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosai

In [6]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v8n_resplit_tl.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v8n_resplit_tl)}")
print(f"Learning Rate inicial: {model_v8n_resplit_tl.trainer.args.lr0}")

Batch Size: 49
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000040
Optimizer: None
Learning Rate inicial: 0.01


In [5]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v8n_resplit_no_tl = f"04_v8n_resplit_no_tl"

print(f"Iniciando entrenamiento: {run_name_v8n_resplit_no_tl}")

model_v8n_resplit_no_tl = YOLO('yolov8n.yaml')

results_v8n_resplit_no_tl = model_v8n_resplit_no_tl.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_v8n_resplit_no_tl,

    epochs=300,            # Ajustable
    patience=20,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=640,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=False,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 04_v8n_resplit_no_tl
New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, momentum=0.937, 

In [7]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v8n_resplit_no_tl.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v8n_resplit_no_tl)}")
print(f"Learning Rate inicial: {model_v8n_resplit_no_tl.trainer.args.lr0}")

Batch Size: 49
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000409
Optimizer: None
Learning Rate inicial: 0.01


In [3]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v11n_resplit_tl = f"05_v11n_resplit_tl"

print(f"Iniciando entrenamiento: {run_name_v11n_resplit_tl}")

model_v11n_resplit_tl = YOLO('yolo11n.pt')
results_v11n_resplit_tl = model_v11n_resplit_tl.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_v11n_resplit_tl,
    epochs=100,            # Ajustable
    patience=20,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=640,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=True,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 05_v11n_resplit_tl
New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosa

In [4]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v11n_resplit_tl.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v11n_resplit_tl)}")
print(f"Learning Rate inicial: {model_v11n_resplit_tl.trainer.args.lr0}")

Batch Size: 43
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000040
Optimizer: None
Learning Rate inicial: 0.01


In [5]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v11n_resplit_no_tl = f"06_v11n_resplit_no_tl"

print(f"Iniciando entrenamiento: {run_name_v11n_resplit_no_tl}")

model_v11n_resplit_no_tl = YOLO('yolo11n.yaml')
results_v11n_resplit_no_tl = model_v11n_resplit_no_tl.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_v11n_resplit_no_tl,
    epochs=300,            # Ajustable
    patience=20,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=640,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=False,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 06_v11n_resplit_no_tl
New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937,

In [6]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v11n_resplit_no_tl.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v11n_resplit_no_tl)}")
print(f"Learning Rate inicial: {model_v11n_resplit_no_tl.trainer.args.lr0}")

Batch Size: 44
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000317
Optimizer: None
Learning Rate inicial: 0.01


In [ ]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v11s_resplit_tl = f"07_v11s_resplit_tl"

print(f"Iniciando entrenamiento: {run_name_v11s_resplit_tl}")

model_v11s_resplit_tl = YOLO('yolo11s.pt')
results_v11s_resplit_tl = model_v11s_resplit_tl.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_v11s_resplit_tl,
    epochs=100,            # Ajustable
    patience=20,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=640,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=True,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 06_v11s_resplit_tl
New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosa

In [4]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v11s_resplit_tl.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v11s_resplit_tl)}")
print(f"Learning Rate inicial: {model_v11s_resplit_tl.trainer.args.lr0}")

Batch Size: 23
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000040
Optimizer: None
Learning Rate inicial: 0.01


In [3]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v11m_resplit_tl = f"08_v11m_resplit_tl"

print(f"Iniciando entrenamiento: {run_name_v11m_resplit_tl}")

model_v11m_resplit_tl = YOLO('yolo11m.pt')
results_v11m_resplit_tl = model_v11m_resplit_tl.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_v11m_resplit_tl,
    epochs=100,            # Ajustable
    patience=20,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=640,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=True,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 08_v11m_resplit_tl
New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosa

In [4]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v11m_resplit_tl.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v11m_resplit_tl)}")
print(f"Learning Rate inicial: {model_v11m_resplit_tl.trainer.args.lr0}")

Batch Size: 10
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000040
Optimizer: None
Learning Rate inicial: 0.01


In [3]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v8n_baseline_tl_1280 = f"09_v8n_baseline_tl_1280"

print(f"Iniciando entrenamiento: {run_name_v8n_baseline_tl_1280}")

# Cargar modelo Nano pre-entrenado
model_v8n_baseline_tl_1280 = YOLO('yolov8n.pt')

results_v8n_baseline_tl_1280 = model_v8n_baseline_tl_1280.train(
    data=YAML_BASELINE,
    project=MODELS_DIR,
    name=run_name_v8n_baseline_tl_1280,

    epochs=300,            # Ajustable
    patience=50,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=1280,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=True,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 09_v8n_baseline_tl_1280
New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas_baseline/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momen

In [5]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v8n_baseline_tl_1280.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v8n_baseline_tl_1280)}")
print(f"Learning Rate inicial: {model_v8n_baseline_tl_1280.trainer.args.lr0}")

Batch Size: 12
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000027
Optimizer: None
Learning Rate inicial: 0.01


In [4]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v8n_baseline_no_tl_1280 = f"10_v8n_baseline_no_tl_1280"

print(f"Iniciando entrenamiento: {run_name_v8n_baseline_no_tl_1280}")

# Cargar modelo Nano pre-entrenado
model_v8n_baseline_no_tl_1280 = YOLO('yolov8n.yaml')

results_v8n_baseline_no_tl_1280 = model_v8n_baseline_no_tl_1280.train(
    data=YAML_BASELINE,
    project=MODELS_DIR,
    name=run_name_v8n_baseline_no_tl_1280,

    epochs=600,            # Ajustable
    patience=50,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=1280,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=False,
    optimizer='auto',
    verbose=True 
)

Iniciando entrenamiento: 10_v8n_baseline_no_tl_1280
New https://pypi.org/project/ultralytics/8.3.236 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas_baseline/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=600, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, 

In [5]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v8n_baseline_no_tl_1280.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v8n_baseline_no_tl_1280)}")
print(f"Learning Rate inicial: {model_v8n_baseline_no_tl_1280.trainer.args.lr0}")

Batch Size: 12
Decisión de 'Auto':
   • Optimizador:   SGD
   • Learning Rate: 0.003433
Optimizer: None
Learning Rate inicial: 0.01


In [3]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v8n_resplit_tl_1280 = f"11_v8n_resplit_tl_1280"

print(f"Iniciando entrenamiento: {run_name_v8n_resplit_tl_1280}")

model_v8n_resplit_tl_1280 = YOLO('yolov8n.pt')

results_v8n_resplit_tl_1280 = model_v8n_resplit_tl_1280.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_v8n_resplit_tl_1280,

    epochs=300,            # Ajustable
    patience=50,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=1280,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=True,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 11_v8n_resplit_tl_1280
New https://pypi.org/project/ultralytics/8.3.236 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937,

In [4]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v8n_resplit_tl_1280.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v8n_resplit_tl_1280)}")
print(f"Learning Rate inicial: {model_v8n_resplit_tl_1280.trainer.args.lr0}")

Batch Size: 12
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000231
Optimizer: None
Learning Rate inicial: 0.01


In [3]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v8n_resplit_no_tl_1280 = f"12_v8n_resplit_no_tl_1280"

print(f"Iniciando entrenamiento: {run_name_v8n_resplit_no_tl_1280}")

model_v8n_resplit_no_tl_1280 = YOLO('yolov8n.yaml')

results_v8n_resplit_no_tl_1280 = model_v8n_resplit_no_tl_1280.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_v8n_resplit_no_tl_1280,

    epochs=600,            # Ajustable
    patience=50,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=1280,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=False,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 12_v8n_resplit_no_tl_1280
New https://pypi.org/project/ultralytics/8.3.236 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=600, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, momentum=0

In [4]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v8n_resplit_no_tl_1280.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v8n_resplit_no_tl_1280)}")
print(f"Learning Rate inicial: {model_v8n_resplit_no_tl_1280.trainer.args.lr0}")

Batch Size: 12
Decisión de 'Auto':
   • Optimizador:   SGD
   • Learning Rate: 0.003598
Optimizer: None
Learning Rate inicial: 0.01


In [3]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v11n_resplit_tl_1280 = f"13_v11n_resplit_tl_1280"

print(f"Iniciando entrenamiento: {run_name_v11n_resplit_tl_1280}")

model_v11n_resplit_tl_1280 = YOLO('yolo11n.pt')
results_v11n_resplit_tl_1280 = model_v11n_resplit_tl_1280.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_v11n_resplit_tl_1280,
    epochs=300,            # Ajustable
    patience=50,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=1280,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=True,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 13_v11n_resplit_tl_1280
New https://pypi.org/project/ultralytics/8.3.237 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937

In [4]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v11n_resplit_tl_1280.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v11n_resplit_tl_1280)}")
print(f"Learning Rate inicial: {model_v11n_resplit_tl_1280.trainer.args.lr0}")

Batch Size: 10
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000145
Optimizer: None
Learning Rate inicial: 0.01


In [5]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v11n_resplit_no_tl_1280 = f"14_v11n_resplit_no_tl_1280"

print(f"Iniciando entrenamiento: {run_name_v11n_resplit_no_tl_1280}")

model_v11n_resplit_no_tl_1280 = YOLO('yolo11n.yaml')
results_v11n_resplit_no_tl_1280 = model_v11n_resplit_no_tl_1280.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_v11n_resplit_no_tl_1280,
    epochs=600,            # Ajustable
    patience=50,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=1280,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=False,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 14_v11n_resplit_no_tl_1280
New https://pypi.org/project/ultralytics/8.3.237 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=600, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=

In [6]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v11n_resplit_no_tl_1280.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v11n_resplit_no_tl_1280)}")
print(f"Learning Rate inicial: {model_v11n_resplit_no_tl_1280.trainer.args.lr0}")

Batch Size: 8
Decisión de 'Auto':
   • Optimizador:   SGD
   • Learning Rate: 0.005132
Optimizer: None
Learning Rate inicial: 0.01


In [3]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v11s_resplit_tl_1280 = f"15_v11s_resplit_tl_1280"

print(f"Iniciando entrenamiento: {run_name_v11s_resplit_tl_1280}")

model_v11s_resplit_tl_1280 = YOLO('yolo11s.pt')
results_v11s_resplit_tl_1280 = model_v11s_resplit_tl_1280.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_v11s_resplit_tl_1280,
    epochs=300,            # Ajustable
    patience=50,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=1280,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=True,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 15_v11s_resplit_tl_1280
New https://pypi.org/project/ultralytics/8.3.237 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937

In [4]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v11s_resplit_tl_1280.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v11s_resplit_tl_1280)}")
print(f"Learning Rate inicial: {model_v11s_resplit_tl_1280.trainer.args.lr0}")

Batch Size: 5
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000027
Optimizer: None
Learning Rate inicial: 0.01


In [3]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v11m_resplit_tl_1280 = f"16_v11m_resplit_tl_1280"

print(f"Iniciando entrenamiento: {run_name_v11m_resplit_tl_1280}")

model_v11m_resplit_tl_1280 = YOLO('yolo11m.pt')
results_v11m_resplit_tl_1280 = model_v11m_resplit_tl_1280.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_v11m_resplit_tl_1280,
    epochs=300,            # Ajustable
    patience=50,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=1280,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=True,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 16_v11m_resplit_tl_1280
New https://pypi.org/project/ultralytics/8.3.237 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937

In [4]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v11m_resplit_tl_1280.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v11m_resplit_tl_1280)}")
print(f"Learning Rate inicial: {model_v11m_resplit_tl_1280.trainer.args.lr0}")

Batch Size: 2
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000027
Optimizer: None
Learning Rate inicial: 0.01


In [7]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v8s_resplit_tl_1280 = f"17_v8s_resplit_tl_1280"

print(f"Iniciando entrenamiento: {run_name_v8s_resplit_tl_1280}")

model_v8s_resplit_tl_1280 = YOLO('yolov8s.pt')
results_v8s_resplit_tl_1280 = model_v8s_resplit_tl_1280.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_v8s_resplit_tl_1280,
    epochs=300,            # Ajustable
    patience=50,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=1280,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=True,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 17_v8s_resplit_tl_1280
New https://pypi.org/project/ultralytics/8.3.237 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937,

In [8]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v8s_resplit_tl_1280.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v8s_resplit_tl_1280)}")
print(f"Learning Rate inicial: {model_v8s_resplit_tl_1280.trainer.args.lr0}")

Batch Size: 6
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000027
Optimizer: None
Learning Rate inicial: 0.01


In [3]:
# Entrenamiento YOLOv8n con dataset Baseline
run_name_v8m_resplit_tl_1280 = f"18_v8m_resplit_tl_1280"

print(f"Iniciando entrenamiento: {run_name_v8m_resplit_tl_1280}")

model_v8m_resplit_tl_1280 = YOLO('yolov8m.pt')
results_v8m_resplit_tl_1280 = model_v8m_resplit_tl_1280.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_v8m_resplit_tl_1280,
    epochs=300,            # Ajustable
    patience=50,          # Early stopping    
    batch=-1,          # Ajustable
    imgsz=1280,            # Tamaño de imagen    
    
    exist_ok=True,         # Sobrescribir si existe
    pretrained=True,
    optimizer='auto',
    verbose=True
)

Iniciando entrenamiento: 18_v8m_resplit_tl_1280
New https://pypi.org/project/ultralytics/8.3.237 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937,

In [4]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v8m_resplit_tl_1280.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v8m_resplit_tl_1280)}")
print(f"Learning Rate inicial: {model_v8m_resplit_tl_1280.trainer.args.lr0}")

Batch Size: 3
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000027
Optimizer: None
Learning Rate inicial: 0.01


In [17]:
# Nombre del experimento
run_name_clean = f"{DATE_STR}_v8n_clean_tl_640"

print(f"Iniciando entrenamiento: {run_name_clean}")

# Cargar modelo Nano pre-entrenado
model_clean = YOLO('yolov8n.pt')

results_clean = model_clean.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_clean,
    epochs=30,            # Ajustable
    imgsz=640,
    batch=-1,          # Ajustable
    patience=10,          # Early stopping
    exist_ok=True,         # Sobrescribir si existe
    verbose=True
)

Iniciando entrenamiento: 20251201_v8n_clean_tl_640
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=20251201_v8n_clean_tl_640, nbs=64, nms=False, opset=None, optimi

In [2]:
# Nombre del experimento
run_name_base = f"{DATE_STR}_v8n_baseline_tl_640"

print(f"Iniciando entrenamiento Baseline: {run_name_base}")

# Cargar modelo nuevo para no contaminar pesos
model_base = YOLO('yolov8n.pt')

results_base = model_base.train(
    data=YAML_BASELINE,
    project=MODELS_DIR,
    name=run_name_base,
    epochs=30,
    imgsz=640,
    batch=-1,
    patience=10,
    exist_ok=True,
    verbose=True
)

Iniciando entrenamiento Baseline: 20251201_v8n_baseline_tl_640
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas_baseline/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=20251201_v8n_baseline_tl_640, nbs=64, nms=F

In [3]:
# Cargar mejores pesos
# Asegúrate que estos nombres coincidan exactamente con los definidos en las celdas de entrenamiento
run_name_clean = f"{DATE_STR}_v8n_clean_tl_640"
run_name_base = f"{DATE_STR}_v8n_baseline_tl_640" # Nota: Ajusta si usaste otro nombre en la celda 3

path_clean_weights = os.path.join(MODELS_DIR, run_name_clean, 'weights', 'best.pt')
path_base_weights = os.path.join(MODELS_DIR, run_name_base, 'weights', 'best.pt')

model_final_clean = YOLO(path_clean_weights)
model_final_base = YOLO(path_base_weights)

print("--- VALIDACION CRUZADA ---")

# 1. Validar Modelo Propuesto (Clean)
# Validamos contra el set de datos limpio (test) y definimos nombre específico
metrics_clean = model_final_clean.val(
    data=YAML_CLEAN, 
    split='test', 
    project=MODELS_DIR,
    name=f"{run_name_clean}_val"  # <--- CAMBIO: Nombre personalizado para la carpeta
)

# 2. Validar Modelo Original (Baseline)
# Usamos el MISMO set de datos limpio como 'Gold Standard' para comparar justamente
metrics_base = model_final_base.val(
    data=YAML_CLEAN, 
    split='test', 
    project=MODELS_DIR,
    name=f"{run_name_base}_val"   # <--- CAMBIO: Nombre personalizado para la carpeta
)

print("\nRESULTADOS COMPARATIVOS (mAP50-95):")
print(f"Modelo Propuesto (Clean Split): {metrics_clean.box.map:.4f}")
print(f"Modelo Original (Raw Split):    {metrics_base.box.map:.4f}")

--- VALIDACION CRUZADA ---
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 689.2±41.7 MB/s, size: 2479.1 KB)
val: Scanning /home/roberto/moca_proyecto/datasets/02_placas/test/labels... 198 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 198/198 3.2Kit/s 0.1s
val: New cache created: /home/roberto/moca_proyecto/datasets/02_placas/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 1.5it/s 8.9s0.3s
                   all        198        242      0.977      0.942      0.985      0.848
Speed: 1.1ms preprocess, 1.8ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /home/roberto/moca_proyecto/models/02_placas/20251201_v8n_clean_tl_640_val
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA G

In [3]:
# Nombre del experimento
run_name_clean = f"{DATE_STR}_v11n_clean_tl_100"

print(f"Iniciando entrenamiento: {run_name_clean}")

# Cargar modelo Nano pre-entrenado
model_clean = YOLO('yolo11n.pt')

results_clean = model_clean.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_clean,
    epochs=100,            # Ajustable
    imgsz=640,
    batch=-1,          # Ajustable
    patience=10,          # Early stopping
    exist_ok=True,         # Sobrescribir si existe
    verbose=True
)

Iniciando entrenamiento: 20251201_v11n_clean_tl_100
New https://pypi.org/project/ultralytics/8.3.234 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.9

In [6]:
import glob
import random

run_name_clean = f"{DATE_STR}_v11n_clean_tl_100_inference"

# Tomar una imagen de prueba aleatoria
test_images = glob.glob('../../datasets/02_placas/test/images/*.jpg')
if test_images:
    sample_img = random.choice(test_images)
    
    # Prediccion
    res = model_clean.predict(sample_img, save=True, project=MODELS_DIR, name=run_name_clean)
     
    print(f"Inferencia guardada en {res[0].save_dir}")
else:
    print("No se encontraron imagenes de prueba.")


image 1/1 /home/roberto/moca_proyecto/notebooks/02_placas/../../datasets/02_placas/test/images/00428.jpg: 448x640 1 license_plate, 39.8ms
Speed: 1.8ms preprocess, 39.8ms inference, 0.9ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /home/roberto/moca_proyecto/models/02_placas/20251201_v11n_clean_tl_100_inference
Inferencia guardada en /home/roberto/moca_proyecto/models/02_placas/20251201_v11n_clean_tl_100_inference


In [8]:
# Entrenamiento con YOLOv8n a 100 épocas
run_name_clean = f"{DATE_STR}_v8n_clean_tl_100"

print(f"Iniciando entrenamiento: {run_name_clean}")

# Cargar modelo Nano pre-entrenado
model_v8n_100 = YOLO('yolov8n.pt')

results_v8n_100 = model_v8n_100.train(
    data=YAML_CLEAN,
    project=MODELS_DIR,
    name=run_name_clean,
    epochs=100,            # Ajustable
    imgsz=640,
    batch=-1,          # Ajustable
    patience=10,          # Early stopping
    exist_ok=True,         # Sobrescribir si existe
    verbose=True
)

Iniciando entrenamiento: 20251201_v8n_clean_tl_100
New https://pypi.org/project/ultralytics/8.3.234 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.93

In [9]:
import glob
import random

run_name_v8n = f"{DATE_STR}_v8n_clean_tl_100_inference"

# Tomar una imagen de prueba aleatoria
test_images = glob.glob('../../datasets/02_placas/test/images/*.jpg')
if test_images:
    sample_img = random.choice(test_images)
    
    # Prediccion
    res = model_v8n_100.predict(sample_img, save=True, project=MODELS_DIR, name=run_name_v8n)
     
    print(f"Inferencia guardada en {res[0].save_dir}")
else:
    print("No se encontraron imagenes de prueba.")


image 1/1 /home/roberto/moca_proyecto/notebooks/02_placas/../../datasets/02_placas/test/images/00149.jpg: 480x640 1 license_plate, 55.4ms
Speed: 3.5ms preprocess, 55.4ms inference, 2.5ms postprocess per image at shape (1, 3, 480, 640)
Results saved to /home/roberto/moca_proyecto/models/02_placas/20251201_v8n_clean_tl_100_inference
Inferencia guardada en /home/roberto/moca_proyecto/models/02_placas/20251201_v8n_clean_tl_100_inference


In [10]:
# Cargar mejores pesos
# Asegúrate que estos nombres coincidan exactamente con los definidos en las celdas de entrenamiento
run_name_v11 = f"{DATE_STR}_v11n_clean_tl_100"
run_name_v8 = f"{DATE_STR}_v8n_clean_tl_100" # Nota: Ajusta si usaste otro nombre en la celda 3

path_v11_weights = os.path.join(MODELS_DIR, run_name_v11, 'weights', 'best.pt')
path_v8_weights = os.path.join(MODELS_DIR, run_name_v8, 'weights', 'best.pt')

model_final_v11 = YOLO(path_v11_weights)
model_final_v8 = YOLO(path_v8_weights)

print("--- VALIDACION CRUZADA ---")

# 1. Validar Modelo YOLOv11n
metrics_v11 = model_final_v11.val(
    data=YAML_CLEAN, 
    split='test', 
    project=MODELS_DIR,
    name=f"{run_name_v11}_val"
)

# 2. Validar Modelo YOLOv8n
metrics_v8 = model_final_v8.val(
    data=YAML_CLEAN, 
    split='test', 
    project=MODELS_DIR,
    name=f"{run_name_v8}_val"
)

print("\nRESULTADOS COMPARATIVOS (mAP50-95):")
print(f"Modelo YOLOv11n: {metrics_v11.box.map:.4f}")
print(f"Modelo YOLOv8n:    {metrics_v8.box.map:.4f}")

--- VALIDACION CRUZADA ---
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 632.5±86.5 MB/s, size: 2479.1 KB)
val: Scanning /home/roberto/moca_proyecto/datasets/02_placas/test/labels.cache... 198 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 198/198 724.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 1.5it/s 8.5s0.3s
                   all        198        242      0.979       0.95      0.987      0.874
Speed: 1.0ms preprocess, 2.4ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /home/roberto/moca_proyecto/models/02_placas/20251201_v11n_clean_tl_100_val
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
Model summary (fused): 72 layers, 3,005,843 par

In [ ]:
print("\nINFORMACION DEL MODELO YOLOv11n:")
# print(model_final_v11.info)
model_final_v11.


INFORMACION DEL MODELO YOLOv11n:
<bound method Model.info of YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1))
          (act): SiLU(inplace=True)
        )
        (m): ModuleList(
          (0): Bottleneck(
            (cv1): Conv(
              (conv): Conv2d(16, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
              (act): SiLU(inplace=True)
            )
            (cv2): Conv(
              (conv): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1)

In [12]:
print("\nINFORMACION DEL MODELO YOLOv8n:")
print(model_final_v8.info)


INFORMACION DEL MODELO YOLOv8n:
<bound method Model.info of YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1))
          (act): SiLU(inplace=True)
        )
        (m): ModuleList(
          (0): Bottleneck(
            (cv1): Conv(
              (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
              (act): SiLU(inplace=True)
            )
            (cv2): Conv(
              (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1)

In [16]:
from torchinfo import summary

def desplegar_arquitectura_completa(modelo_yolo, input_size=(1, 3, 640, 640)):
    """
    Muestra el resumen completo de capas, params y tamaños de memoria.
    input_size: (Batch, Canales, Alto, Ancho)
    """
    print(f"\n🔍 ARQUITECTURA DETALLADA: {modelo_yolo.task_map}")
    # Accedemos al modelo interno de PyTorch (modelo.model)
    summary(modelo_yolo.model, 
            input_size=input_size, 
            col_names=["input_size", "output_size", "num_params", "kernel_size", "mult_adds"],
            verbose=1)

In [17]:
desplegar_arquitectura_completa(model_final_v8)


🔍 ARQUITECTURA DETALLADA: {'classify': {'model': <class 'ultralytics.nn.tasks.ClassificationModel'>, 'trainer': <class 'ultralytics.models.yolo.classify.train.ClassificationTrainer'>, 'validator': <class 'ultralytics.models.yolo.classify.val.ClassificationValidator'>, 'predictor': <class 'ultralytics.models.yolo.classify.predict.ClassificationPredictor'>}, 'detect': {'model': <class 'ultralytics.nn.tasks.DetectionModel'>, 'trainer': <class 'ultralytics.models.yolo.detect.train.DetectionTrainer'>, 'validator': <class 'ultralytics.models.yolo.detect.val.DetectionValidator'>, 'predictor': <class 'ultralytics.models.yolo.detect.predict.DetectionPredictor'>}, 'segment': {'model': <class 'ultralytics.nn.tasks.SegmentationModel'>, 'trainer': <class 'ultralytics.models.yolo.segment.train.SegmentationTrainer'>, 'validator': <class 'ultralytics.models.yolo.segment.val.SegmentationValidator'>, 'predictor': <class 'ultralytics.models.yolo.segment.predict.SegmentationPredictor'>}, 'pose': {'model'

In [18]:

desplegar_arquitectura_completa(model_final_v11)


🔍 ARQUITECTURA DETALLADA: {'classify': {'model': <class 'ultralytics.nn.tasks.ClassificationModel'>, 'trainer': <class 'ultralytics.models.yolo.classify.train.ClassificationTrainer'>, 'validator': <class 'ultralytics.models.yolo.classify.val.ClassificationValidator'>, 'predictor': <class 'ultralytics.models.yolo.classify.predict.ClassificationPredictor'>}, 'detect': {'model': <class 'ultralytics.nn.tasks.DetectionModel'>, 'trainer': <class 'ultralytics.models.yolo.detect.train.DetectionTrainer'>, 'validator': <class 'ultralytics.models.yolo.detect.val.DetectionValidator'>, 'predictor': <class 'ultralytics.models.yolo.detect.predict.DetectionPredictor'>}, 'segment': {'model': <class 'ultralytics.nn.tasks.SegmentationModel'>, 'trainer': <class 'ultralytics.models.yolo.segment.train.SegmentationTrainer'>, 'validator': <class 'ultralytics.models.yolo.segment.val.SegmentationValidator'>, 'predictor': <class 'ultralytics.models.yolo.segment.predict.SegmentationPredictor'>}, 'pose': {'model'